# 스마트 창고 출고 지연 예측
향후 30분간 평균 출고 지연 시간(분) 예측  
LightGBM + CatBoost + XGBoost 앙상블

## 1. 라이브러리 및 환경 설정

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from scipy.optimize import minimize
import warnings, gc
warnings.filterwarnings('ignore')

N_SPLITS = 5
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

## 2. 데이터 로드 및 레이아웃 병합

In [ ]:
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')
layout = pd.read_csv('./data/layout_info.csv')

train = train.merge(layout, on='layout_id', how='left')
test = test.merge(layout, on='layout_id', how='left')

train = train.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
test = test.sort_values(['scenario_id', 'ID']).reset_index(drop=True)

train['time_idx'] = train.groupby('scenario_id').cumcount()
test['time_idx'] = test.groupby('scenario_id').cumcount()

combined_lt = pd.concat([train['layout_type'], test['layout_type']], axis=0).astype(str)
lt_map = {v: i for i, v in enumerate(combined_lt.unique())}
train['layout_type'] = train['layout_type'].astype(str).map(lt_map)
test['layout_type'] = test['layout_type'].astype(str).map(lt_map)

print(f'train: {train.shape}  test: {test.shape}')

## 3. 결측값 처리 (median 대체 + 결측 플래그)

In [ ]:
all_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
miss_cols = [c for c in all_cols if train[c].isnull().any() or test[c].isnull().any()]

for col in miss_cols:
    train[f'{col}_isna'] = train[col].isna().astype(np.int8)
    test[f'{col}_isna'] = test[col].isna().astype(np.int8)

train['num_missing'] = train[miss_cols].isnull().sum(axis=1)
test['num_missing'] = test[miss_cols].isnull().sum(axis=1)

medians = train[miss_cols].median()
train[miss_cols] = train[miss_cols].fillna(medians)
test[miss_cols] = test[miss_cols].fillna(medians)
train = train.fillna(0)
test = test.fillna(0)

print(f'결측 컬럼 {len(miss_cols)}개 처리 완료')

## 4. 파생변수 - 비율 / 교차 피처

In [ ]:
def add_ratio_features(df):
    df['order_per_robot'] = df['order_inflow_15m'] / (df['robot_active'] + 1)
    df['charge_pressure'] = df['robot_charging'] / (df['charger_count'] + 1)
    df['congestion_per_robot'] = df['congestion_score'] / (df['robot_active'] + 1)
    df['orders_per_pack_station'] = df['order_inflow_15m'] / (df['pack_station_count'] + 1)
    df['active_per_aisle_width'] = df['robot_active'] / (df['aisle_width_avg'] + 1)
    df['congestion_per_aisle'] = df['congestion_score'] / (df['aisle_width_avg'] + 1)
    df['charging_per_robot'] = df['robot_charging'] / (df['robot_active'] + 1)
    df['low_battery_pressure'] = df['low_battery_ratio'] * df['robot_active']
    df['pack_pressure'] = df['pack_utilization'] / (df['pack_station_count'] + 1)
    df['intersection_pressure'] = df['congestion_score'] * df['intersection_count']
    df['robot_density'] = df['robot_total'] / (df['floor_area_sqm'] + 1)
    df['charger_per_robot'] = df['charger_count'] / (df['robot_total'] + 1)
    df['congestion_x_robot_active'] = df['congestion_score'] * df['robot_active']
    df['order_inflow_x_congestion'] = df['order_inflow_15m'] * df['congestion_score']
    df['low_battery_x_robot_active'] = df['low_battery_ratio'] * df['robot_active']
    df['pack_util_x_order_inflow'] = df['pack_utilization'] * df['order_inflow_15m']
    df['robot_idle_ratio'] = df['robot_idle'] / (df['robot_total'] + 1)
    df['effective_robots'] = df['robot_active'] - df['robot_charging']
    df['order_per_effective_robot'] = df['order_inflow_15m'] / df['effective_robots'].clip(lower=1)
    df['battery_risk'] = df['low_battery_ratio'] * df['charge_queue_length']
    df['congestion_battery_interaction'] = df['congestion_score'] * df['low_battery_ratio']
    df['utilization_congestion'] = df['robot_utilization'] * df['congestion_score']
    df['dock_congestion'] = df['loading_dock_util'] * df['congestion_score']
    lt = df['layout_type'].astype(float)
    df['layout_x_congestion'] = lt * df['congestion_score']
    df['layout_x_order_inflow'] = lt * df['order_inflow_15m']
    df['layout_x_robot_active'] = lt * df['robot_active']
    return df

train = add_ratio_features(train)
test = add_ratio_features(test)
print(f'비율/교차 피처 추가 → train {train.shape}')

## 5. 파생변수 - 시계열 피처 (lag / diff / rolling / expanding / EWM)

In [ ]:
def add_time_features(df):
    G = 'scenario_id'

    key_cols = [
        'order_inflow_15m', 'congestion_score', 'robot_charging',
        'battery_mean', 'pack_utilization', 'low_battery_ratio',
        'robot_utilization', 'max_zone_density', 'robot_idle',
        'charge_queue_length',
    ]
    for col in key_cols:
        grp = df.groupby(G)[col]
        df[f'{col}_lag1'] = grp.shift(1)
        df[f'{col}_lag2'] = grp.shift(2)
        df[f'{col}_lag3'] = grp.shift(3)
        df[f'{col}_diff1'] = df[col] - grp.shift(1)
        df[f'{col}_diff2'] = (
            df[f'{col}_diff1']
            - df.groupby(G)[f'{col}_diff1'].shift(1)
        )

    rolling_cols = [
        'order_inflow_15m', 'congestion_score', 'robot_charging',
        'battery_mean', 'low_battery_ratio', 'pack_utilization',
    ]
    for col in rolling_cols:
        shifted = df.groupby(G)[col].shift(1)
        for w in [2, 3, 5]:
            df[f'{col}_rmean_{w}'] = shifted.groupby(df[G]).transform(
                lambda x: x.rolling(w, min_periods=1).mean())
        df[f'{col}_rstd_3'] = shifted.groupby(df[G]).transform(
            lambda x: x.rolling(3, min_periods=1).std())
        df[f'{col}_rmax_3'] = shifted.groupby(df[G]).transform(
            lambda x: x.rolling(3, min_periods=1).max())
        df[f'{col}_rmin_3'] = shifted.groupby(df[G]).transform(
            lambda x: x.rolling(3, min_periods=1).min())

    expand_cols = [
        'order_inflow_15m', 'congestion_score', 'low_battery_ratio',
        'robot_utilization', 'battery_mean', 'pack_utilization',
        'charge_queue_length', 'max_zone_density',
    ]
    for col in expand_cols:
        shifted = df.groupby(G)[col].shift(1)
        df[f'{col}_exp_mean'] = shifted.groupby(df[G]).transform(
            lambda x: x.expanding(min_periods=1).mean())
        df[f'{col}_exp_std'] = shifted.groupby(df[G]).transform(
            lambda x: x.expanding(min_periods=1).std())

    ewm_cols = [
        'order_inflow_15m', 'congestion_score', 'low_battery_ratio',
        'robot_utilization', 'battery_mean', 'pack_utilization',
    ]
    for col in ewm_cols:
        shifted = df.groupby(G)[col].shift(1)
        for sp in [2, 4]:
            df[f'{col}_ewm_{sp}'] = shifted.groupby(df[G]).transform(
                lambda x: x.ewm(span=sp, min_periods=1).mean())

    df = df.fillna(0)
    return df

train = add_time_features(train)
test = add_time_features(test)
print(f'시계열 피처 추가 → train {train.shape}')

## 6. Target Encoding (layout / time_idx / layout x time)

In [ ]:
y_full = train[TARGET]
groups_full = train['scenario_id']
kf_te = GroupKFold(n_splits=N_SPLITS)

for stat_name, stat_func in [('mean', 'mean'), ('std', 'std'), ('median', 'median')]:
    col_name = f'layout_te_{stat_name}'
    train[col_name] = np.nan
    for tr_idx, va_idx in kf_te.split(train, y_full, groups=groups_full):
        agg = train.iloc[tr_idx].groupby('layout_id')[TARGET].agg(stat_func)
        train.loc[train.index[va_idx], col_name] = (
            train.iloc[va_idx]['layout_id'].map(agg).values
        )
    test[col_name] = test['layout_id'].map(
        train.groupby('layout_id')[TARGET].agg(stat_func)
    )

for stat_name, stat_func in [('mean', 'mean'), ('std', 'std')]:
    col_name = f'time_te_{stat_name}'
    agg = train.groupby('time_idx')[TARGET].agg(stat_func)
    train[col_name] = train['time_idx'].map(agg)
    test[col_name] = test['time_idx'].map(agg)

cross = train.groupby(['layout_id', 'time_idx'])[TARGET].mean().reset_index()
cross.columns = ['layout_id', 'time_idx', 'layout_time_te_mean']
train = train.merge(cross, on=['layout_id', 'time_idx'], how='left')
test = test.merge(cross, on=['layout_id', 'time_idx'], how='left')

train = train.fillna(0)
test = test.fillna(0)
print(f'Target Encoding 추가 → train {train.shape}')

## 7. 피처 선택 (LightGBM importance 기준 TOP-K)

In [ ]:
y = train[TARGET]
groups = train['scenario_id']

exclude = set(ID_COLS + [TARGET])
candidate_cols = [c for c in train.columns if c not in exclude]

TOP_K = 120

kf_sel = GroupKFold(n_splits=5)
imp_acc = np.zeros(len(candidate_cols))
X_cand = train[candidate_cols]

for fold, (tr, va) in enumerate(kf_sel.split(X_cand, y, groups=groups)):
    if fold >= 3:
        break
    m = LGBMRegressor(
        n_estimators=800, learning_rate=0.05, max_depth=7,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1,
        random_state=42 + fold, verbose=-1,
    )
    m.fit(
        X_cand.iloc[tr], np.log1p(y.iloc[tr]),
        eval_set=[(X_cand.iloc[va], np.log1p(y.iloc[va]))],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
    )
    imp_acc += m.feature_importances_

imp_acc /= 3
top_idx = np.argsort(imp_acc)[::-1][:TOP_K]
feature_cols = [candidate_cols[i] for i in top_idx]

X = train[feature_cols]
X_test = test[feature_cols]
sw = (1.0 + np.log1p(y.values)) * np.where(y.values > 30, 2.0, 1.0)

del X_cand
gc.collect()

print(f'선택 피처 {len(feature_cols)}개')
print(f'top-10: {feature_cols[:10]}')

## 8. 모델 학습 함수 정의

In [ ]:
def run_lgb(X, y, X_test, groups, params,
            sw=None, use_log=True, label='LGB'):
    kf = GroupKFold(n_splits=N_SPLITS)
    oof = np.zeros(len(X))
    tp = np.zeros(len(X_test))
    for fold, (tr, va) in enumerate(kf.split(X, y, groups=groups)):
        yt = np.log1p(y.iloc[tr]) if use_log else y.iloc[tr]
        yv = np.log1p(y.iloc[va]) if use_log else y.iloc[va]
        swt = sw[tr] if sw is not None else None
        m = LGBMRegressor(**params)
        m.fit(
            X.iloc[tr], yt, sample_weight=swt,
            eval_set=[(X.iloc[va], yv)],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
        )
        pv = m.predict(X.iloc[va])
        pt = m.predict(X_test)
        if use_log:
            pv, pt = np.expm1(pv), np.expm1(pt)
        oof[va] = np.clip(pv, 0, None)
        tp += np.clip(pt, 0, None) / N_SPLITS
    mae = mean_absolute_error(y, oof)
    print(f'  [{label}] OOF MAE = {mae:.4f}')
    return oof, tp, mae


def run_cat(X, y, X_test, groups, params,
            sw=None, use_log=True, label='Cat'):
    kf = GroupKFold(n_splits=N_SPLITS)
    oof = np.zeros(len(X))
    tp = np.zeros(len(X_test))
    for fold, (tr, va) in enumerate(kf.split(X, y, groups=groups)):
        yt = np.log1p(y.iloc[tr]) if use_log else y.iloc[tr]
        yv = np.log1p(y.iloc[va]) if use_log else y.iloc[va]
        swt = sw[tr] if sw is not None else None
        m = CatBoostRegressor(**params)
        fit_kw = dict(
            eval_set=(X.iloc[va], yv),
            use_best_model=True,
            early_stopping_rounds=100,
        )
        if swt is not None:
            fit_kw['sample_weight'] = swt
        m.fit(X.iloc[tr], yt, **fit_kw)
        pv = m.predict(X.iloc[va])
        pt = m.predict(X_test)
        if use_log:
            pv, pt = np.expm1(pv), np.expm1(pt)
        oof[va] = np.clip(pv, 0, None)
        tp += np.clip(pt, 0, None) / N_SPLITS
    mae = mean_absolute_error(y, oof)
    print(f'  [{label}] OOF MAE = {mae:.4f}')
    return oof, tp, mae


def run_xgb(X, y, X_test, groups, params,
            sw=None, use_log=True, label='XGB'):
    kf = GroupKFold(n_splits=N_SPLITS)
    oof = np.zeros(len(X))
    tp = np.zeros(len(X_test))
    for fold, (tr, va) in enumerate(kf.split(X, y, groups=groups)):
        yt = np.log1p(y.iloc[tr]) if use_log else y.iloc[tr]
        yv = np.log1p(y.iloc[va]) if use_log else y.iloc[va]
        swt = sw[tr] if sw is not None else None
        m = xgb.XGBRegressor(**params)
        m.fit(
            X.iloc[tr], yt, sample_weight=swt,
            eval_set=[(X.iloc[va], yv)], verbose=False,
        )
        pv = m.predict(X.iloc[va])
        pt = m.predict(X_test)
        if use_log:
            pv, pt = np.expm1(pv), np.expm1(pt)
        oof[va] = np.clip(pv, 0, None)
        tp += np.clip(pt, 0, None) / N_SPLITS
    mae = mean_absolute_error(y, oof)
    print(f'  [{label}] OOF MAE = {mae:.4f}')
    return oof, tp, mae

## 9. LightGBM 학습 (3개 config)

In [ ]:
all_oof = {}
all_test = {}

lgb_cfgs = {
    'lgb_l2': {
        'n_estimators': 5000, 'learning_rate': 0.02,
        'max_depth': 8, 'num_leaves': 127, 'min_child_samples': 20,
        'subsample': 0.8, 'colsample_bytree': 0.7,
        'reg_alpha': 0.3, 'reg_lambda': 0.3,
        'random_state': 42, 'verbose': -1,
    },
    'lgb_deep': {
        'n_estimators': 5000, 'learning_rate': 0.02,
        'max_depth': 10, 'num_leaves': 255, 'min_child_samples': 30,
        'subsample': 0.75, 'colsample_bytree': 0.65,
        'reg_alpha': 0.5, 'reg_lambda': 0.5,
        'random_state': 42, 'verbose': -1,
    },
    'lgb_huber': {
        'n_estimators': 5000, 'learning_rate': 0.02,
        'max_depth': 8, 'num_leaves': 127, 'min_child_samples': 20,
        'objective': 'huber', 'huber_delta': 1.0,
        'subsample': 0.8, 'colsample_bytree': 0.7,
        'reg_alpha': 0.3, 'reg_lambda': 0.3,
        'random_state': 42, 'verbose': -1,
    },
}

for name, p in lgb_cfgs.items():
    o, t, _ = run_lgb(X, y, X_test, groups, p, sw=sw, label=name)
    all_oof[name] = o
    all_test[name] = t

## 10. CatBoost 학습 (3개 config)

In [ ]:
cat_cfgs = {
    'cat_mae': {
        'iterations': 5000, 'learning_rate': 0.02, 'depth': 8,
        'loss_function': 'MAE', 'eval_metric': 'MAE',
        'l2_leaf_reg': 3.0, 'random_seed': 42, 'verbose': 0,
    },
    'cat_rmse': {
        'iterations': 5000, 'learning_rate': 0.02, 'depth': 8,
        'loss_function': 'RMSE', 'eval_metric': 'MAE',
        'l2_leaf_reg': 3.0, 'random_seed': 42, 'verbose': 0,
    },
    'cat_deep': {
        'iterations': 5000, 'learning_rate': 0.02, 'depth': 10,
        'loss_function': 'MAE', 'eval_metric': 'MAE',
        'l2_leaf_reg': 5.0, 'random_seed': 42, 'verbose': 0,
    },
}

for name, p in cat_cfgs.items():
    o, t, _ = run_cat(X, y, X_test, groups, p, sw=sw, label=name)
    all_oof[name] = o
    all_test[name] = t

## 11. XGBoost 학습 (2개 config)

In [ ]:
xgb_cfgs = {
    'xgb': {
        'n_estimators': 5000, 'learning_rate': 0.02,
        'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.7,
        'reg_alpha': 0.5, 'reg_lambda': 1.0, 'min_child_weight': 10,
        'tree_method': 'hist', 'eval_metric': 'mae',
        'early_stopping_rounds': 100,
        'random_state': 42, 'verbosity': 0,
    },
    'xgb_deep': {
        'n_estimators': 5000, 'learning_rate': 0.02,
        'max_depth': 10, 'subsample': 0.75, 'colsample_bytree': 0.65,
        'reg_alpha': 1.0, 'reg_lambda': 2.0, 'min_child_weight': 15,
        'tree_method': 'hist', 'eval_metric': 'mae',
        'early_stopping_rounds': 100,
        'random_state': 42, 'verbosity': 0,
    },
}

for name, p in xgb_cfgs.items():
    o, t, _ = run_xgb(X, y, X_test, groups, p, sw=sw, label=name)
    all_oof[name] = o
    all_test[name] = t

## 12. 개별 모델 결과 비교

In [ ]:
print('=== 개별 모델 OOF MAE ===')
for name, oof_i in sorted(all_oof.items(),
                           key=lambda x: mean_absolute_error(y, x[1])):
    print(f'  {name}: {mean_absolute_error(y, oof_i):.4f}')

## 13. 최적 가중치 앙상블 (Nelder-Mead)

In [ ]:
oof_arr = np.array(list(all_oof.values()))
test_arr = np.array(list(all_test.values()))
names = list(all_oof.keys())
n_models = len(names)

def mae_obj(w):
    w = np.abs(w)
    w = w / w.sum()
    return mean_absolute_error(y, (w[:, None] * oof_arr).sum(axis=0))

best_mae = 1e9
best_w = None
for trial in range(30):
    w0 = (np.ones(n_models) / n_models if trial == 0
          else np.random.dirichlet(np.ones(n_models)))
    res = minimize(mae_obj, w0, method='Nelder-Mead',
                   options={'maxiter': 10000, 'xatol': 1e-7, 'fatol': 1e-7})
    if res.fun < best_mae:
        best_mae = res.fun
        best_w = np.abs(res.x) / np.abs(res.x).sum()

opt_oof = (best_w[:, None] * oof_arr).sum(axis=0)
opt_test = (best_w[:, None] * test_arr).sum(axis=0)

print(f'최적 가중치 앙상블 MAE: {best_mae:.4f}')
for nm, w in zip(names, best_w):
    if w > 0.01:
        print(f'  {nm}: {w:.4f}')

## 14. LightGBM Stacking (2단계 메타 모델)

In [ ]:
stk_tr = pd.DataFrame(all_oof)
stk_te = pd.DataFrame(all_test)

stk_params = {
    'n_estimators': 1000, 'learning_rate': 0.02,
    'max_depth': 4, 'num_leaves': 15,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 1.0, 'reg_lambda': 1.0, 'verbose': -1,
}

kf_stk = GroupKFold(n_splits=N_SPLITS)
stk_oof = np.zeros(len(X))
stk_tp = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(kf_stk.split(stk_tr, y, groups=groups)):
    m = LGBMRegressor(**stk_params)
    m.fit(
        stk_tr.iloc[tr], y.iloc[tr],
        eval_set=[(stk_tr.iloc[va], y.iloc[va])],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
    )
    stk_oof[va] = np.clip(m.predict(stk_tr.iloc[va]), 0, None)
    stk_tp += np.clip(m.predict(stk_te), 0, None) / N_SPLITS

stk_mae = mean_absolute_error(y, stk_oof)
print(f'LGB Stacking MAE: {stk_mae:.4f}')

## 15. 최종 모델 선택 및 결과 비교

In [ ]:
candidates = {
    'opt_weight_ensemble': (opt_oof, opt_test, best_mae),
    'simple_avg': (
        np.mean(list(all_oof.values()), axis=0),
        np.mean(list(all_test.values()), axis=0),
        mean_absolute_error(y, np.mean(list(all_oof.values()), axis=0)),
    ),
    'lgb_stacking': (stk_oof, stk_tp, stk_mae),
}
for nm in all_oof:
    m = mean_absolute_error(y, all_oof[nm])
    candidates[nm] = (all_oof[nm], all_test[nm], m)

print('=== 최종 후보 (MAE 오름차순) ===')
for nm, (_, _, m) in sorted(candidates.items(), key=lambda x: x[1][2]):
    print(f'  {nm}: {m:.4f}')

best_name = min(candidates, key=lambda k: candidates[k][2])
final_oof, final_test_preds, final_mae = candidates[best_name]
print(f'\n>>> 최종 선택: {best_name}')
print(f'>>> OOF MAE: {final_mae:.4f}')

## 16. 제출 파일 저장

In [ ]:
submission = pd.DataFrame({
    'ID': test['ID'],
    TARGET: final_test_preds,
})
submission.to_csv('./submission.csv', index=False)

print('submission.csv 저장 완료')
print(f'shape: {submission.shape}')
print(submission.head())
print(f'min={submission[TARGET].min():.4f}  max={submission[TARGET].max():.4f}')